#  LLMpy

Beyond all the hype, large language models are a powerful tool in the modern data science toolkit.
dsllmpy` (lumpy? yeah, let's go with that) is a small utility package that makes using LLMs in data science work a little bit easier.

The workhorse of LLM-powered data science is the ability to take unstructured text data
and turn it into useable data, using simple labels, or fancy structured output schemas.
dsllmpy` makes doing this in parallel, over a whole column of values, easy.

In [ ]:
# Setup
import os
from dotenv import load_dotenv
from dsllmpy import OpenAIClient
from pydantic import BaseModel

load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
llm_client = OpenAIClient(api_key=OPENAI_API_KEY, model="gpt-5.4-mini")

In [ ]:
# Get some data
from datasets import load_dataset
import itertools
import pandas as pd

def get_n_rows_from_hf(dataset: str, n_rows: int):
    ds = load_dataset(dataset, streaming=True, split="train")
    return pd.DataFrame(itertools.islice(ds, n_rows))


review_df = get_n_rows_from_hf("Yelp/yelp_review_full", 20)
for txt in review_df['text'].iloc[:2]:
    print(txt)

## Example 1: Simple text output

In [ ]:
system_prompt = """
Classify the following yelp review as either 'Positive', 'Negative', or 'Neutral'.
Output only the label.
""".strip()

# Classify a single value
one_result = llm_client.call(system_prompt=system_prompt, user_prompt=review_df["text"].iloc[0])
print(one_result)

In [ ]:
# Classify everything in parallel
results = await llm_client.call_many(
    system_prompt=system_prompt, user_prompt=review_df["text"], max_requests_per_minute=100
)
print(results)

### Example 2: Structured outputs

In [ ]:
# Classification restricting output to valid options
from typing import Literal

class Sentiment(BaseModel):
    value: Literal['Positive', 'Negative', 'Neutral']

structured_results = await llm_client.call_many(
    system_prompt=system_prompt, user_prompt=review_df["text"],
    max_requests_per_minute=100,
    response_format=Sentiment
)
print(structured_results)

In [ ]:
results = [r.value for r in structured_results]
print(results)

In [ ]:
# Richer structure
from pydantic import Field
class ReviewAnnotations(BaseModel):
    subject_name: str = Field(description="Name of the person or place being reviewed")
    subject_type: str | None = Field(description="What kind of person/place is being reviewed? Leave blank if unknown")
    sentiment: Literal['Positive', 'Negative', 'Neutral']

structured_results = await llm_client.call_many(
    system_prompt="Extract the information required", user_prompt=review_df["text"],
    max_requests_per_minute=100,
    response_format=ReviewAnnotations
)
result_df = pd.DataFrame([r.model_dump() for r in structured_results])
result_df['text'] = review_df['text']
result_df


## Example 3: Tagging

In [ ]:

soflow_df = get_n_rows_from_hf("mirzaei2114/stackoverflowVQA", 20)
soflow_df['Text'] = soflow_df['Title'] + '\n\n' + soflow_df['Body']
soflow_df = soflow_df[['Text', 'Tags']]
soflow_df.head()


In [ ]:
possible_tags = soflow_df['Tags'].explode().unique().tolist()
print(possible_tags)



In [ ]:
class QuestionTags(BaseModel):
    tags: list[Literal[*possible_tags]] | None # type: ignore

tagging_results = await llm_client.call_many(
    "Select all the tags that apply to this question. Leave blank if none apply",
    soflow_df['Text'],
    response_format=QuestionTags
)
soflow_df['inferred_tags'] = [r.tags for r in tagging_results]
soflow_df

Easy!